# Céleste 레이어 분리 (See-through)

원화 한 장을 **앞머리·뒷머리·얼굴·눈·옷** 같은 부위별 레이어(PSD)로 나눕니다. 가려진 부분도 채워 줍니다.

**쓰는 법**
1. 위 메뉴 **런타임 → 런타임 유형 변경 → T4 GPU** 선택 후 저장
2. **런타임 → 모두 실행** (처음엔 설치·모델 내려받기로 10~15분)
3. 마지막 셀이 끝나면 `celeste_layers.zip` 이 자동으로 내려받아집니다 → 그 파일을 Claude에게 올려 주세요

중간에 빨간 오류가 나면 그 화면을 캡처해서 보내 주세요.

In [ ]:
# 1) GPU 확인 — 'Tesla T4' 같은 이름이 나와야 합니다
!nvidia-smi --query-gpu=name,memory.total,compute_cap --format=csv

In [ ]:
# 2) See-through 코드 받기
import os, shutil
os.chdir('/content')                                   # 항상 바깥 폴더에서 시작
shutil.rmtree('/content/st', ignore_errors=True)
!git clone -q --depth 1 https://huggingface.co/spaces/24yearsold/see-through-demo /content/st
!ln -sf common/assets /content/st/assets
print('코드 받기 성공' if os.path.isdir('/content/st/common') else '코드 받기 실패')

In [ ]:
# 3) 필요한 라이브러리 설치 (Colab에 있는 torch는 그대로 씀)
#    빨간 글씨로 "pip's dependency resolver does not currently take into account..." 가 나와도 정상입니다.
#    맨 아래에 버전 목록이 찍히면 설치 성공 → 다음 셀로 넘어가세요.
import subprocess, sys
def pip(*pk):
    return subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-U', *pk]).returncode == 0
# See-through가 맞춰 둔 torch 2.8 (CUDA 12.8)로 맞춘다. Colab 기본(최신 CUDA 13)은 T4에서 계산 라이브러리가 시작되지 않을 수 있음
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'torch==2.8.0', 'torchvision==0.23.0',
                '--index-url', 'https://download.pytorch.org/whl/cu128'])
common = ['safetensors', 'einops', 'pycocotools', 'scikit-learn', 'opencv-python-headless', 'pyyaml']
pip(*common)
# See-through가 맞춰 둔 버전으로 고정 (최신 버전은 내부 사용법이 달라 실패할 수 있음)
if not pip('psd-tools[composite]==1.14.2'): pip('psd-tools[composite]')
if not pip('bitsandbytes==0.49.2'): pip('bitsandbytes')
if not pip('accelerate==1.13.0'): pip('accelerate')
# See-through가 쓰는 버전을 먼저 시도하고, 안 되면 최신 버전으로
if not pip('diffusers==0.37.0'): pip('diffusers')
if not pip('transformers==5.0.0', 'huggingface-hub==1.7.2'): pip('transformers', 'huggingface-hub')
import importlib.metadata as md
for n in ['torch', 'diffusers', 'transformers', 'huggingface-hub', 'accelerate', 'bitsandbytes', 'psd-tools']:
    try: print(f'{n:16s}', md.version(n))
    except Exception as e: print(f'{n:16s} 설치 안 됨  ← 이 줄을 캡처해서 보내 주세요')

In [ ]:
# 4) 입력 그림 받기 (Céleste 상반신, 정사각형)
!wget -q -O /content/input.png https://raw.githubusercontent.com/vecar0/midnight/celeste-concept/work/celeste/seethrough_input.png
from PIL import Image; im = Image.open('/content/input.png'); print(im.size); im.resize((384,384))

In [ ]:
# 5) 레이어 분리 실행 (4비트 압축 모델, T4 메모리에 맞춤)
import os, subprocess, shutil
# 혹시 이전 시도에서 코드가 바뀌었을 수 있으니 깨끗하게 다시 받는다
os.chdir('/content')      # 지울 폴더 안에 서 있으면 다시 받기가 실패하므로 먼저 밖으로
shutil.rmtree('/content/st', ignore_errors=True); shutil.rmtree('/content/out', ignore_errors=True)
g = subprocess.run(['git', 'clone', '-q', '--depth', '1', 'https://huggingface.co/spaces/24yearsold/see-through-demo', '/content/st'],
                   capture_output=True, text=True)
assert os.path.isdir('/content/st'), 'See-through 코드 받기 실패: ' + g.stderr
subprocess.run('ln -sf common/assets assets', shell=True, cwd='/content/st')
os.environ['PYTHONPATH'] = '/content/st/common:/content/st'
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
print(subprocess.run(['nvidia-smi', '--query-gpu=name,memory.used,memory.total', '--format=csv'], capture_output=True, text=True).stdout)
print(subprocess.run(['python', '-c', 'import torch; print("torch", torch.__version__, "GPU", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")'],
                     capture_output=True, text=True).stdout)
if True:
    r = subprocess.run(['python', 'inference/scripts/inference_psd_quantized.py',
                        '--srcp', '/content/input.png', '--save_dir', '/content/out',
                        '--resolution', '1024', '--save_to_psd', '--tblr_split'],
                       cwd='/content/st', capture_output=True, text=True)
print('종료 코드:', r.returncode, '(−9 이면 메모리 부족으로 강제 종료)')
print('--- 출력 끝부분 ---'); print(r.stdout[-2000:]); print('--- 오류 끝부분 ---'); print(r.stderr[-4000:])
print('성공!' if os.path.exists('/content/out/input.psd') else '실패 → 다음 셀(6)을 실행하세요')

### 6) 5번이 실패했을 때만
T4 GPU는 일부 계산 형식(bf16)을 지원하지 않아 실패할 수 있습니다. 이 셀은 계산 형식을 T4가 지원하는 것으로 바꿔 다시 시도합니다. **5번이 '성공!'이면 건너뛰세요.**

In [ ]:
if not os.path.exists('/content/out/input.psd'):
    !grep -rl "bfloat16" /content/st/common /content/st/inference/scripts | xargs sed -i 's/torch.bfloat16/torch.float16/g'
    r = subprocess.run(['python', 'inference/scripts/inference_psd_quantized.py',
                        '--srcp', '/content/input.png', '--save_dir', '/content/out',
                        '--resolution', '1024', '--save_to_psd', '--tblr_split'],
                       cwd='/content/st', capture_output=True, text=True)
    print('종료 코드:', r.returncode, '(−9 이면 메모리 부족으로 강제 종료)')
    print('--- 출력 끝부분 ---'); print(r.stdout[-2000:]); print('--- 오류 끝부분 ---'); print(r.stderr[-4000:])
print('성공!' if os.path.exists('/content/out/input.psd') else '여전히 실패 — 이 셀 아래 글자를 복사해서 보내 주세요')

In [ ]:
# 7) 결과 미리보기 + 내려받기
import glob, shutil, matplotlib.pyplot as plt
pngs = [p for p in sorted(glob.glob('/content/out/input/*.png')) if not p.endswith('_depth.png')]
print(len(pngs), '개 레이어')
cols = 6; rows = (len(pngs)+cols-1)//cols
plt.figure(figsize=(18, 3*rows))
for i, p in enumerate(pngs):
    plt.subplot(rows, cols, i+1); plt.imshow(Image.open(p)); plt.title(os.path.basename(p)[:-4], fontsize=8); plt.axis('off')
plt.show()
shutil.make_archive('/content/celeste_layers', 'zip', '/content/out')
from google.colab import files; files.download('/content/celeste_layers.zip')